# 01 API Collector — Graphic Novels

Goal:
Collect a large raw dataset of graphic novel / comics-related books using the Open Library API.
Pull as many graphic novel/comics candidates as possible for now, filter later.

Output:
`../Data/Raw/graphic_novels/openlibrary_graphic_novels_raw.csv`

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

In [ ]:
# Create directories for raw and clean data
raw_dir = Path("../Data/Raw/graphic_novels")
raw_dir.mkdir(parents=True, exist_ok=True)

clean_dir = Path("../Data/Clean")
clean_dir.mkdir(parents=True, exist_ok=True)

In [11]:
# Define search terms related to graphic novels
search_terms = [
    "graphic novel",
    "graphic novels",
    "sequential art",
    "graphic memoir",
    "literary comics",
    "adult graphic novel",
    "illustrated novel",
    "visual storytelling",
    "graphic biography",
    "graphic history"
]

In [12]:
# Function to fetch books from Open Library API based on search term and number of pages
def fetch_openlibrary_books(search_term, pages=10):
    books = []

    for page in range(1, pages + 1):
        url = "https://openlibrary.org/search.json"
        params = {
            "q": search_term,
            "page": page
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Failed for {search_term}, page {page}")
            continue

        data = response.json()
        docs = data.get("docs", [])

        for doc in docs:
            books.append({
                "search_term": search_term,
                "ol_key": doc.get("key"),
                "title": doc.get("title"),
                "author": ", ".join(doc.get("author_name", [])) if doc.get("author_name") else None,
                "first_publish_year": doc.get("first_publish_year"),
                "publisher": ", ".join(doc.get("publisher", [])[:3]) if doc.get("publisher") else None,
                "language": ", ".join(doc.get("language", [])[:5]) if doc.get("language") else None,
                "subject": ", ".join(doc.get("subject", [])[:30]) if doc.get("subject") else None,
                "isbn": doc.get("isbn", [None])[0] if doc.get("isbn") else None,
                "cover_i": doc.get("cover_i")
            })

        print(f"Fetched {len(docs)} books for '{search_term}' page {page}")
        time.sleep(0.5)

    return books

In [13]:
# Collect books for all search terms and save to CSV
all_books = []

for term in search_terms:
    books = fetch_openlibrary_books(term, pages=10)
    all_books.extend(books)

raw_df = pd.DataFrame(all_books)

raw_df.shape

Fetched 100 books for 'graphic novel' page 1
Fetched 100 books for 'graphic novel' page 2
Fetched 100 books for 'graphic novel' page 3
Fetched 100 books for 'graphic novel' page 4
Fetched 100 books for 'graphic novel' page 5
Fetched 100 books for 'graphic novel' page 6
Fetched 100 books for 'graphic novel' page 7
Fetched 100 books for 'graphic novel' page 8
Fetched 100 books for 'graphic novel' page 9
Fetched 100 books for 'graphic novel' page 10
Fetched 100 books for 'graphic novels' page 1
Fetched 100 books for 'graphic novels' page 2
Fetched 100 books for 'graphic novels' page 3
Fetched 100 books for 'graphic novels' page 4
Fetched 100 books for 'graphic novels' page 5
Fetched 100 books for 'graphic novels' page 6
Fetched 100 books for 'graphic novels' page 7
Fetched 100 books for 'graphic novels' page 8
Fetched 100 books for 'graphic novels' page 9
Fetched 100 books for 'graphic novels' page 10
Fetched 87 books for 'sequential art' page 1
Fetched 0 books for 'sequential art' page 2

(7642, 10)

In [14]:
raw_df.head()

,search_term,ol_key,title,author,first_publish_year,publisher,language,subject,isbn,cover_i
0,graphic novel,/works/OL24244816W,Allergic,"Megan Wagner Lloyd, Michelle Mee Nutter",2021.0,None,eng,None,None,10724638.0
1,graphic novel,/works/OL24793569W,The Alchemist Graphic Novel,Paulo Coelho,2010.0,None,eng,None,None,11556106.0
2,graphic novel,/works/OL15902631W,The Kite Runner--the graphic novel,Khaled Hosseini,2011.0,None,"eng, ara",None,None,8063795.0
3,graphic novel,/works/OL16793507W,Drama,Raina Telgemeier,2012.0,None,"spa, fre, eng",None,None,12360868.0
4,graphic novel,/works/OL28062430W,Moon Rising,Tui T. Sutherland,2022.0,None,eng,None,None,15099568.0


In [15]:
# Remove duplicates based on title and author
raw_df = raw_df.drop_duplicates(subset=["title", "author"])

raw_df.shape

(5609, 10)

In [16]:
# Save raw data to CSV
output_path = raw_dir / "openlibrary_graphic_novels_raw.csv"

raw_df.to_csv(output_path, index=False)

print(f"Saved raw data to: {output_path}")
print(f"Rows saved: {len(raw_df)}")

Saved raw data to: ../Data/Raw/graphic_novels/openlibrary_graphic_novels_raw.csv
Rows saved: 5609
